# Liquid Neural Network (LNN) - Liquid Time-Constant network
"Liquid Time-constant Networks" (Hasani et al., 2021), inspired by the nervous system of the worm *C. elegans*

**Idea:** each neuron is a small differential equation (like a leaky capacitor). Its **time constant** (how fast it reacts vs. how long it remembers) is not fixed but depends on the current input: the dynamics are "liquid". The ODE is solved with a few cheap numerical steps per input step. LNNs are very parameter-efficient and handle irregularly sampled time series well.

### LNN vs RNN vs SSM
| | RNN / LSTM | SSM (Mamba) | LNN (LTC) |
|---|---|---|---|
| Time | discrete steps | continuous ODE, discretized with $\Delta_t$ | continuous ODE, solved with $K$ solver steps |
| Recurrence | nonlinear update | **linear** in the state | nonlinear, **input-dependent time constant** |
| State bounded? | by tanh | by $\bar A < 1$ | yes, provably (between 0 and $A$) |
| Typical size | 100s-1000s of units | large | **tiny** (tens of neurons) |
| Strengths | general sequences | long sequences, LMs | control, robotics, irregular time series |

**Task:** sequential MNIST. Read the image **one row per time step** (28 steps of 28 pixels) and classify the digit from the final state, so the model has to integrate information over time.

Shape flow: `image (B, 28, 28) -> loop t over rows: I_t (B, 28), x (B, H) -> ODE solver K steps -> x_T (B, H) -> logits (B, 10)`

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

## 1. Data - MNIST as sequences
Each image `(1, 28, 28)` becomes a sequence of 28 rows, each a 28-dim input vector.

In [ ]:
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_ds = datasets.MNIST("../data", train=True, download=True, transform=tf)
test_ds = datasets.MNIST("../data", train=False, download=True, transform=tf)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=512, shuffle=False, num_workers=2, pin_memory=True)

img, label = train_ds[0]
print(img.shape, "-> sequence", img[0].shape, "(T=28 steps, 28 inputs each), label", label)

## 2. LTC cell
Each hidden neuron $x_i$ follows the ODE below. We integrate it over one input step with $K = 6$ **fused semi-implicit Euler** steps (the solver from the paper), which stays stable even for stiff dynamics.

### Formula: LTC dynamics
$$\frac{dx(t)}{dt} = -\left[\frac{1}{\tau} + f(x(t), I(t))\right] \odot x(t) + f(x(t), I(t)) \odot A$$
- $x(t) \in \mathbb{R}^{H}$: hidden state (neuron "potentials"), $H = 64$
- $I(t) \in \mathbb{R}^{28}$: input at time $t$ (one image row)
- $\tau \in \mathbb{R}^{H}_{>0}$: base time constant per neuron (`softplus(tau_raw)`)
- $f(x, I) = \sigma(W [I, x] + b) \in (0, 1)^H$: input- and state-dependent "synaptic" activation (`self.f`)
- $A \in \mathbb{R}^{H}$: bias / "reversal potential" each neuron is pulled toward when $f$ is active
- first term: leak toward 0; second term: drive toward $A$

### Formula: Liquid time constant
$$\tau_{sys} = \frac{\tau}{1 + \tau\, f(x, I)}$$
- the effective time constant; it changes with the input at every moment
- $f \approx 0$: $\tau_{sys} = \tau$, slow, the neuron holds its memory
- $f$ large: $\tau_{sys}$ small, the neuron reacts quickly to the current input

### Formula: Fused ODE solver step
$$x_{k+1} = \frac{x_k + \delta\, f(x_k, I)\odot A}{1 + \delta\left(\frac{1}{\tau} + f(x_k, I)\right)}, \qquad \delta = \frac{\Delta t}{K}$$
- explicit Euler for the drive term, implicit for the leak term (the division)
- $K = 6$ unfolds per input step, $\Delta t = 1$
- the denominator is always $> 1$, so the state can't blow up

In [ ]:
class LTCCell(nn.Module):
    # I_t (B, In), x (B, H) -> x_new (B, H)
    def __init__(self, input_size, hidden_size, unfolds=6):
        super().__init__()
        self.unfolds = unfolds
        self.f = nn.Linear(input_size + hidden_size, hidden_size)   # W, b of the synaptic activation
        self.tau_raw = nn.Parameter(torch.zeros(hidden_size))       # tau = softplus(tau_raw) > 0
        self.A = nn.Parameter(torch.randn(hidden_size))             # reversal potentials

    def gate(self, I, x):
        return torch.sigmoid(self.f(torch.cat([I, x], dim=1)))     # (B, H) f(x, I)

    def forward(self, I, x, dt=1.0):
        tau = F.softplus(self.tau_raw)                              # (H,)
        delta = dt / self.unfolds
        for _ in range(self.unfolds):                               # K solver steps per input step
            f = self.gate(I, x)                                     # (B, H)
            x = (x + delta * f * self.A) / (1 + delta * (1 / tau + f))  # (B, H)
        return x

## 3. LTC classifier
Run the cell over the 28 rows, then a linear readout from the final state.

### Formula: Readout
$$\hat y = \text{softmax}(W_{out}\, x_T + b_{out})$$
- $x_T \in \mathbb{R}^{H}$: state after the last row ($T = 28$)
- $W_{out} \in \mathbb{R}^{10 \times H}$: `self.head`

In [ ]:
class LTCClassifier(nn.Module):
    # images (B, 1, 28, 28) -> logits (B, 10)
    def __init__(self, input_size=28, hidden_size=64, num_classes=10):
        super().__init__()
        self.hidden_size = hidden_size
        self.cell = LTCCell(input_size, hidden_size)
        self.head = nn.Linear(hidden_size, num_classes)

    def forward(self, images):
        seq = images[:, 0]                                        # (B, 28 rows, 28 pixels)
        x = torch.zeros(seq.shape[0], self.hidden_size, device=seq.device)
        for t in range(seq.shape[1]):
            x = self.cell(seq[:, t], x)                           # (B, H)
        return self.head(x)                                       # (B, 10)

model = LTCClassifier().to(device)
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e3:.1f}K")
print(model(torch.randn(2, 1, 28, 28, device=device)).shape)

## 4. Training
Standard classification: cross-entropy + Adam, a few epochs.

### Formula: Cross-entropy
$$\mathcal{L} = -\frac{1}{B}\sum_{b=1}^{B} \log \hat y_{b, c_b}$$
- $\hat y_{b, c}$: predicted probability of class $c$ for sample $b$
- $c_b$: the true digit

In [ ]:
epochs = 5
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()     # (B, 10) -> (B,)
        total += y.size(0)
    model.train()
    return correct / total

for epoch in range(epochs):
    total_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)                     # (B, 1, 28, 28), (B,)
        loss = F.cross_entropy(model(x), y)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    print(f"epoch {epoch+1} | loss {total_loss / len(train_loader):.4f} | test acc {evaluate(model, test_loader):.4f}")

## 5. Seeing the "liquid" time constants
Feed one digit row by row and record $\tau_{sys} = \tau / (1 + \tau f)$ for a few neurons. Fixed-time-constant models (RNN, SSM with fixed $A$) would give flat lines; here each neuron speeds up or slows down depending on what it is currently reading.

In [ ]:
model.eval()
img, label = test_ds[0]
seq = img[0].to(device)[None]                                   # (1, 28, 28)
cell = model.cell
tau = F.softplus(cell.tau_raw)
x = torch.zeros(1, model.hidden_size, device=device)
taus = []
with torch.no_grad():
    for t in range(28):
        f = cell.gate(seq[:, t], x)                              # (1, H) at the start of this row
        taus.append((tau / (1 + tau * f))[0].cpu())               # (H,)
        x = cell(seq[:, t], x)
taus = torch.stack(taus)                                         # (28, H)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].imshow(img[0], cmap="gray"); axes[0].set_title(f"input (label {label})"); axes[0].axis("off")
axes[1].plot(taus[:, :8]); axes[1].set_xlabel("row (time step)"); axes[1].set_ylabel(r"$\tau_{sys}$")
axes[1].set_title("effective time constant, 8 neurons")
plt.tight_layout(); plt.show()

## Notes
- 64 neurons (~6K params) should reach high-90s% test accuracy on row-by-row MNIST.
- Cost: $K = 6$ solver steps per input step makes LTCs slower than an LSTM of the same size. **CfC** networks (Hasani et al., 2022) replace the solver with a closed-form approximation and are much faster.
- Where LNNs shine: small models controlling robots/drones, and time series sampled at irregular times (pass the real gap as `dt`).